# Обучение модели ruBERT для классификации новостей

## 1. Установка зависимостей

In [ ]:
!pip install -q torch transformers datasets scikit-learn matplotlib seaborn pandas joblib tqdm

## 2. Импорты и настройки

In [ ]:
import os
import json
import re
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from transformers import BertTokenizerFast, BertModel
from datasets import load_dataset

# Настройка устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

## 3. Конфигурация

In [ ]:
# Конфигурация
CONFIG = {
    # Датасет
    "dataset_name": "data-silence/rus_news_classifier",

    "class_names": [
        "climate", "conflicts", "culture", "economy", "gloss",
        "health", "politics", "science", "society", "sports", "travel"
    ],
    
    # BERT
    "bert_model_name": "ai-forever/ruBERT-base",
    "max_length": 128,
    "batch_size": 8,
    "num_epochs": 3,
    "learning_rate": 2e-5,
    "dropout": 0.3,
    
    # Пути для сохранения
    "output_dir": "./bert_model_output",
    "plots_dir": "./bert_plots",
    "metrics_file": "./bert_metrics.json"
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["plots_dir"], exist_ok=True)
print(f"Классов: {len(CONFIG['class_names'])}")

## 4. Предобработка данных

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('\u2028', ' ')
    text = text.replace('\u2029', ' ')
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s.,!?;:—«»"\'()]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.strip()

print("Загрузка датасета...")
dataset = load_dataset(CONFIG["dataset_name"])

train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

print("Очистка текстов...")
train_df['news'] = train_df['news'].apply(clean_text)
test_df['news'] = test_df['news'].apply(clean_text)

train_df = train_df[train_df['news'].str.len() > 0].copy()
test_df = test_df[test_df['news'].str.len() > 0].copy()

print(f"Обучающих примеров: {len(train_df)}")
print(f"Тестовых примеров: {len(test_df)}")
print(f"Распределение классов в обучающей выборке:")
print(train_df['labels'].value_counts().sort_index())

## 5. Создание DataLoader

In [ ]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

tokenizer = BertTokenizerFast.from_pretrained(CONFIG["bert_model_name"])

train_dataset = NewsDataset(
    train_df['news'].values,
    train_df['labels'].values,
    tokenizer,
    CONFIG["max_length"]
)

test_dataset = NewsDataset(
    test_df['news'].values,
    test_df['labels'].values,
    tokenizer,
    CONFIG["max_length"]
)

# DataLoader
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=0
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0
)

print(f"Загружено батчей в train: {len(train_loader)}")
print(f"Загружено батчей в test: {len(test_loader)}")

## 6. Модель ruBERT

In [ ]:
class ruBERT(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_token)
        x = self.classifier(x)
        return x

# Инициализация модели
model = ruBERT(
    model_name=CONFIG["bert_model_name"],
    num_classes=len(CONFIG["class_names"]),
    dropout=CONFIG["dropout"]
).to(device)

print(f"Модель создана и загружена на {device}")
print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")

## 7. Обучение

In [ ]:
# Loss и Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=CONFIG["learning_rate"])

# История обучения
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": []
}

num_epochs = CONFIG["num_epochs"]
best_val_acc = 0
best_model_state = None

print("\n" + "="*60)
print("Начало обучения...")
print("="*60)

for epoch in range(num_epochs):
    # Обучение
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = total_loss / len(train_loader)
    train_acc = correct / total * 100
    
    # Валидация
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_loss = val_loss / len(test_loader)
    val_acc = val_correct / val_total * 100
    val_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val F1: {val_f1:.4f}")

print("\nОбучение завершено!")
print(f"Лучшая Val Acc: {best_val_acc:.2f}%")

## 8. Сохранение модели

In [ ]:
# Загрузка лучшей модели
if best_model_state is not None:
    model.load_state_dict(best_model_state)

# Сохранение модели
model_path = os.path.join(CONFIG["output_dir"], "bert_model.pth")
tokenizer_path = os.path.join(CONFIG["output_dir"], "tokenizer")

torch.save(model.state_dict(), model_path)
model.bert.save_pretrained(tokenizer_path)
tokenizer.save_pretrained(tokenizer_path)

print(f"Модель сохранена: {model_path}")
print(f"Токенизатор сохранен: {tokenizer_path}")

## 9. Финальная оценка

In [ ]:
# Финальная оценка на тестовом наборе
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Расчет метрик
report = classification_report(all_labels, all_preds, target_names=CONFIG["class_names"], output_dict=True)

final_metrics = {
    "accuracy": report["accuracy"],
    "macro_precision": report["macro avg"]["precision"],
    "macro_recall": report["macro avg"]["recall"],
    "macro_f1": report["macro avg"]["f1-score"],
    "weighted_f1": report["weighted avg"]["f1-score"],
    "per_class": report
}

# Сохранение метрик
with open(CONFIG["metrics_file"], 'w', encoding='utf-8') as f:
    json.dump(final_metrics, f, indent=4, ensure_ascii=False)

print("="*60)
print("Финальные метрики на тестовом наборе:")
print("="*60)
print(f"Accuracy:        {final_metrics['accuracy']:.4f} ({final_metrics['accuracy']*100:.2f}%)")
print(f"Macro Precision: {final_metrics['macro_precision']:.4f}")
print(f"Macro Recall:    {final_metrics['macro_recall']:.4f}")
print(f"Macro F1:        {final_metrics['macro_f1']:.4f}")
print(f"Weighted F1:     {final_metrics['weighted_f1']:.4f}")
print(f"\nМетрики сохранены: {CONFIG['metrics_file']}")

## 10. Визуализация результатов

In [ ]:
# Графики обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
epochs = range(1, len(history["train_loss"]) + 1)
ax1.plot(epochs, history["train_loss"], label="Train Loss", marker="o")
ax1.plot(epochs, history["val_loss"], label="Val Loss", marker="s")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(epochs, history["train_acc"], label="Train Acc", marker="o")
ax2.plot(epochs, history["val_acc"], label="Val Acc", marker="s")
ax2.plot(epochs, history["val_f1"], label="Val F1", marker="^")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Training & Validation Accuracy and F1")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plots_path = os.path.join(CONFIG["plots_dir"], "training_curves.png")
plt.savefig(plots_path, dpi=150)
print(f"Графики сохранены: {plots_path}")
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CONFIG["class_names"], yticklabels=CONFIG["class_names"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

cm_path = os.path.join(CONFIG["plots_dir"], "confusion_matrix.png")
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
print(f"Confusion Matrix сохранена: {cm_path}")
plt.show()

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=CONFIG["class_names"]))

## 11. Финальные результаты

In [ ]:
print("="*60)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ")
print("="*60)
print(f"\nОбучение завершено!")
print(f"\nСохраненные файлы:")
print(f"   Модель:       {CONFIG['output_dir']}/")
print(f"      - bert_model.pth")
print(f"      - tokenizer/")
print(f"   Графики:      {CONFIG['plots_dir']}/")
print(f"      - training_curves.png")
print(f"      - confusion_matrix.png")
print(f"   Метрики:      {CONFIG['metrics_file']}")
print(f"\nЛучшая Val Acc: {best_val_acc:.2f}%")
print(f"Test Accuracy:   {final_metrics['accuracy']*100:.2f}%")
print(f"Test Weighted F1: {final_metrics['weighted_f1']:.4f}")